In [68]:
import numpy as np

import pandas as pd
from sklearn.model_selection import train_test_split
import sklearn
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LinearRegression
from sklearn.svm import SVR
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score
from sklearn.model_selection import learning_curve

from feature_engine.datetime import DatetimeFeatures



import joblib

import matplotlib.pyplot as plt

In [69]:
pd.set_option("display.max_columns", None)
sklearn.set_config(transform_output="default")

In [70]:
train_df = pd.read_csv("data/train.csv")
val_df = pd.read_csv("data/val.csv")
test_df = pd.read_csv("data/test.csv")

In [71]:
train_df.head()
train_full=pd.concat([train_df, val_df])

In [72]:
def split_data(data):
	X = data.drop(columns="price")
	y = data.price.copy()
	return (X, y)
x_test,y_test=split_data(test_df)

df

In [73]:

dt_cols = ["date_of_journey", "dep_time", "arrival_time"]

num_cols = ["duration", "total_stops"]

cat_cols = [col for col in X_train.columns if (col not in dt_cols) and (col not in num_cols)]

In [74]:
num_transformer = Pipeline(steps=[
	("imputer", SimpleImputer(strategy="median")),
	("scaler", StandardScaler())
])

cat_transformer = Pipeline(steps=[
	("imputer", SimpleImputer(strategy="most_frequent")),
	("encoder", OneHotEncoder(sparse_output=False, handle_unknown="ignore"))
])

doj_transformer = Pipeline(steps=[
	("imputer", SimpleImputer(strategy="most_frequent")),
	("extractor", DatetimeFeatures(features_to_extract=["month", "week", "day_of_week", "day_of_month"], format="mixed")),
	("scaler", StandardScaler())
])

time_transformer = Pipeline(steps=[
	("imputer", SimpleImputer(strategy="most_frequent")),
	("extractor", DatetimeFeatures(features_to_extract=["hour", "minute"], format="mixed")),
	("scaler", StandardScaler())
])

In [75]:
preprocessor = ColumnTransformer(transformers=[
	("num", num_transformer, num_cols),
	("cat", cat_transformer, cat_cols),
	("doj", doj_transformer, ["date_of_journey"]),
	("time", time_transformer, ["dep_time", "arrival_time"])
])

In [76]:
x_train,y_train=split_data(train_full)
x=preprocessor.fit_transform(x_train)


array([[-1.0841459 , -1.19152348,  0.        , ..., -0.19128273,
        -0.34015327,  1.4888104 ],
       [ 1.48919359,  0.3186457 ,  0.        , ..., -1.2735047 ,
        -0.9211153 ,  0.88519783],
       [ 1.88431265,  1.82881487,  0.        , ..., -0.73239372,
        -0.34015327, -1.22744616],
       ...,
       [-1.09427716, -1.19152348,  0.        , ..., -0.19128273,
         1.25749232,  1.18700412],
       [ 1.23591214,  0.3186457 ,  0.        , ..., -1.2735047 ,
        -0.19491276,  0.58339155],
       [-1.0841459 , -1.19152348,  0.        , ..., -0.73239372,
        -0.48539378,  0.88519783]])

In [124]:
algorithms={
    "LinearRegression": LinearRegression(),
    "SVR":SVR(),
    "RandomForestRegressor":RandomForestRegressor(n_estimators=10),

}

In [146]:
max=0
namealg=""
for name,alg in algorithms.items():
    model = Pipeline(steps=[
	("pre", preprocessor),
	(name, alg)
     ])
    model.fit(x_train, y_train)
    if r2_score(y_test, model.predict(x_test)) > max:
        max=r2_score(y_test, model.predict(x_test))
        namealg=name


namealg

'LinearRegression()'

In [171]:

rfg=RandomForestRegressor(n_estimators=40)
x_train_after=preprocessor.fit_transform(x_train)
rfg.fit(x_train_after,y_train)
x_test_after=preprocessor.transform(x_test)
r2_score(y_test, rfg.predict(x_test_after))


0.7455849426543346

In [173]:
r2_score(y_train, rfg.predict(x_train_after))

0.9613637146116695